# Scratch Analysis — Lightweight, Rerunnable from the Submission ZIP

This notebook is **FloPy-free** and reruns from the `exports/` bundle **alone** — no
MODFLOW, no FloPy, no course repo, no heavy model workspaces. Each group member takes
one **card** and produces the figures/tables for it from the frozen exports the
steward created with `steward_export_lightweight.ipynb`.

**How to use**
1. Set `STUDENT_NAME` and pick your `CARD` below.
2. Run all cells. Only the section for your `CARD` does work; the rest are inert.
3. Your outputs are saved under `figures/` and `tables/`.
4. Write 2–4 sentences in the final **Interpretation & contribution** section.

**Cards** — one owner each, no duplicates unless the steward approves:

| Card | Topic | Needs |
|------|-------|-------|
| A | Drawdown affected area | base + wells heads |
| B | Pathlines (optional) | `pathlines_summary.csv` (skips if absent) |
| C | Scenario comparison | scenario heads (degrades to A if absent) |
| D | Budget / river exchange | `flow_budget_summary.csv` |
| E | Transport breakthrough | transport CSV + meta |
| F | Provenance / submission QA | `run_info.json` + manifest |

> Do **not** import `flopy`, `pyemu`, or any `_SUPPORT` helper here — that would break
> the "reruns from ZIP" promise. Only `scratch_io`, plus numpy/pandas/geopandas/matplotlib.

In [ ]:
# --- Setup (FloPy-free) ---
import scratch_io                      # template-local reader, sits next to this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

scratch_io.assert_no_flopy()           # guard #1: prove the kernel is FloPy-free

# ---------------------------------------------------------------------
#  >>> EDIT THESE TWO LINES <<<
STUDENT_NAME = 'First Last'
CARD = 'A'                              # one of: A B C D E F
# ---------------------------------------------------------------------

EXPORTS = scratch_io.find_exports()     # locate exports/ (group folder or extracted ZIP)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
TAB = Path('tables');  TAB.mkdir(exist_ok=True)

info = scratch_io.load_run_info(EXPORTS)
scratch_io.check_schema(info)
avail = scratch_io.available_exports(EXPORTS)

print(f'Student : {STUDENT_NAME}')
print(f'Card    : {CARD}')
print(f'Group   : {info.get("group_number")}   schema {info.get("schema_version")}')
print(f'Exports : {EXPORTS}')
print('Present :', sorted(k for k, v in avail.items() if v is not None))

## Card A — Drawdown affected area (base → wells)

Loads the base (no-wells) and wells head fields, computes cell-by-cell drawdown, maps
it, and reports the area where drawdown exceeds 0.5 m.

In [ ]:
if CARD == 'A':
    base  = scratch_io.load_heads_gpkg('base',  EXPORTS)
    wells = scratch_io.load_heads_gpkg('wells', EXPORTS)
    dd = scratch_io.compute_drawdown(base, wells)

    thr = 0.5
    area = scratch_io.affected_area(dd, threshold=thr)
    print(f'Area with drawdown > {thr} m: {area:,.0f} m^2  ({area/1e4:.2f} ha)')
    print(f'Max drawdown: {dd["drawdown_m"].max():.2f} m')

    fig, ax = plt.subplots(figsize=(9, 8))
    dd.plot(column='drawdown_m', cmap='viridis_r', legend=True, ax=ax,
            legend_kwds={'label': 'Drawdown [m]'})
    dd[dd['drawdown_m'] > thr].boundary.plot(ax=ax, color='red', linewidth=0.3)
    ax.set_title(f'Card A — Drawdown (base → wells)\narea > {thr} m = {area/1e4:.2f} ha')
    ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]'); ax.set_aspect('equal')
    fig.tight_layout(); fig.savefig(FIG / 'cardA_drawdown_map.png', dpi=150)
    plt.show()

    summary = pd.DataFrame({
        'metric': ['max_drawdown_m', 'mean_drawdown_m', f'area_gt_{thr}m_m2', f'area_gt_{thr}m_ha'],
        'value':  [dd['drawdown_m'].max(), dd['drawdown_m'].mean(), area, area/1e4],
    })
    summary.to_csv(TAB / 'cardA_drawdown_summary.csv', index=False)
    display(summary)
else:
    print('Card A not selected — skipped.')

## Card B — Pathlines (optional)

Summarises MODPATH particle travel times, if the group exported them. Skips cleanly
when `pathlines_summary.csv` is absent (MODPATH is an optional step).

In [ ]:
if CARD == 'B':
    try:
        pl = scratch_io.load_pathlines_summary(EXPORTS)
    except FileNotFoundError as exc:
        print(f'Card B skipped: {exc}')
    else:
        display(pl)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(pl['metric'], pl['value'], color='#1f77b4')
        ax.set_title('Card B — MODPATH pathline summary')
        ax.set_ylabel('value'); plt.xticks(rotation=30, ha='right')
        fig.tight_layout(); fig.savefig(FIG / 'cardB_pathlines_summary.png', dpi=150)
        plt.show()
        pl.to_csv(TAB / 'cardB_pathlines_summary.csv', index=False)
else:
    print('Card B not selected — skipped.')

## Card C — Scenario comparison

Compares the forcing-scenario head field against the base case. If scenario heads are
absent, it **degrades to an A-style** base→wells comparison and prints a note (so the
card still produces a deliverable).

In [ ]:
if CARD == 'C':
    base = scratch_io.load_heads_gpkg('base', EXPORTS)
    if avail['flow_heads_scenario'] is not None:
        other = scratch_io.load_heads_gpkg('scenario', EXPORTS)
        label, note = 'base → scenario', 'scenario forcing effect'
    else:
        other = scratch_io.load_heads_gpkg('wells', EXPORTS)
        label, note = 'base → wells (DEGRADED)', 'scenario heads absent; showing wells effect instead'
        print('NOTE: no scenario export found — degrading to a base→wells comparison.')

    diff = scratch_io.compute_drawdown(base, other)   # base - other (head change)
    print(f'{note}: max |head change| = {diff["drawdown_m"].abs().max():.2f} m')

    fig, ax = plt.subplots(figsize=(9, 8))
    vmax = float(diff['drawdown_m'].abs().max()) or 1.0
    diff.plot(column='drawdown_m', cmap='RdBu', vmin=-vmax, vmax=vmax, legend=True, ax=ax,
              legend_kwds={'label': 'Head change [m] (base − compare)'})
    ax.set_title(f'Card C — {label}')
    ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]'); ax.set_aspect('equal')
    fig.tight_layout(); fig.savefig(FIG / 'cardC_scenario_head_change.png', dpi=150)
    plt.show()
    diff[['cellid', 'head_base_m', 'head_compare_m', 'drawdown_m']].to_csv(
        TAB / 'cardC_head_change.csv', index=False)
else:
    print('Card C not selected — skipped.')

## Card D — Budget / river exchange

Reads the tidy flow budget and reports net river exchange per model (positive = net
gain of water to the aquifer).

In [ ]:
if CARD == 'D':
    budget = scratch_io.load_budget_summary(EXPORTS)
    riv = scratch_io.river_exchange(budget)
    if riv.empty:
        print('No river-leakage terms in the exported budget (CHD-bounded submodels only).')
        display(budget)
    else:
        display(riv)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(riv['model'], riv['net_m3_d'], color='#2ca02c')
        ax.axhline(0, color='k', lw=0.8)
        ax.set_title('Card D — Net river exchange per model')
        ax.set_ylabel('net river exchange [m³/d]'); plt.xticks(rotation=20, ha='right')
        fig.tight_layout(); fig.savefig(FIG / 'cardD_river_exchange.png', dpi=150)
        plt.show()
        riv.to_csv(TAB / 'cardD_river_exchange.csv', index=False)
    budget.to_csv(TAB / 'cardD_budget_summary.csv', index=False)
else:
    print('Card D not selected — skipped.')

## Card E — Transport breakthrough

Plots the concentration breakthrough at the monitoring well against the regulatory
threshold, and reports the peak and (if any) exceedance time. Requires the transport
exports; prints a note and skips if the group did not run transport.

In [ ]:
if CARD == 'E':
    if avail['transport_breakthrough'] is None:
        print('Card E skipped: no transport_breakthrough.csv (group did not run transport).')
    else:
        bt = scratch_io.load_transport_breakthrough(EXPORTS)
        meta = scratch_io.load_transport_meta(EXPORTS)
        thr = meta.get('threshold_mg_L')
        peak = meta.get('peak_mg_L'); t_peak = meta.get('t_peak_days')
        t_exc = meta.get('t_exceedance_days')
        print(f"Contaminant: {meta.get('contaminant')} (concession {meta.get('concession')})")
        print(f'Peak {peak:.4g} mg/L @ {t_peak:.0f} d | threshold {thr} mg/L | '
              f"exceedance {('t=%.0f d' % t_exc) if t_exc is not None else 'none'}")

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(bt['time_days'], bt['concentration_mg_L'], color='#d62728', lw=2, label='C at monitoring well')
        if thr is not None:
            ax.axhline(thr, color='gray', ls='--', label=f'threshold {thr:g} mg/L')
        if peak is not None and t_peak is not None:
            ax.plot(t_peak, peak, 'ko', label=f'peak {peak:.3g} mg/L')
        ax.set_xlabel('Time [d]'); ax.set_ylabel('Concentration [mg/L]')
        ax.set_title(f"Card E — Breakthrough ({meta.get('contaminant')})")
        ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout(); fig.savefig(FIG / 'cardE_breakthrough.png', dpi=150)
        plt.show()
        bt.to_csv(TAB / 'cardE_breakthrough.csv', index=False)
else:
    print('Card E not selected — skipped.')

## Card F — Provenance / submission QA

Validates the manifest, lists optional artifacts that are absent, and prints a
submission checklist so the group knows what is in the bundle.

In [ ]:
if CARD == 'F':
    scratch_io.check_schema(info)      # raises if schema is incompatible
    print('schema_version :', info.get('schema_version'))
    print('generated_utc  :', info.get('generated_utc'))
    print('group_number   :', info.get('group_number'))
    print('CRS            :', info.get('crs'))
    print('grid           :', info.get('grid'))
    print('head_masking   :', info.get('head_masking'))
    print('package_versions:', info.get('package_versions'))
    print('missing_optional:', info.get('missing_optional'))

    required = ['flow_heads_sub_base.gpkg', 'flow_heads_sub_wells.gpkg', 'flow_budget_summary.csv']
    manifest = info.get('exports', {})
    rows = []
    for fname in ['run_info.json'] + required + [
            'flow_heads_sub_scenario.gpkg', 'transport_breakthrough.csv',
            'transport_meta.json', 'pathlines_summary.csv']:
        present = (EXPORTS / fname).is_file()
        rows.append({'file': fname, 'present': present,
                     'required': fname in required or fname == 'run_info.json'})
    qa = pd.DataFrame(rows)
    display(qa)
    qa.to_csv(TAB / 'cardF_submission_qa.csv', index=False)

    missing_required = qa[(qa['required']) & (~qa['present'])]
    if len(missing_required):
        print('FAIL — missing REQUIRED exports:', list(missing_required['file']))
    else:
        print('OK — all required exports present.')
else:
    print('Card F not selected — skipped.')

## Interpretation & contribution

*Write 2–4 sentences here (edit this markdown cell):*

- **What my card shows:** …
- **Physical / practical reading:** signal vs numerical artefact vs instability? …
- **My contribution to the group deliverable:** …

In [ ]:
# Guard #2: confirm nothing pulled in FloPy/pyemu during the analysis.
scratch_io.assert_no_flopy()
print('OK — scratch analysis stayed FloPy-free and reruns from the exports bundle alone.')